# Assignment 2A — Prompt Engineering & Preference Alignment

For CPT + QLoRA + prompting + DPO in **one** Run all, open `Assignment.ipynb` instead.

| S. No | Name | Bits ID |
|-------|------|---------|
| 1 | Madhu Gururajan | 2025cs05027 |
| 2 | Anand Bharti | 2025cs05036 |

One notebook, **Runtime → Run all**. There are no `RUN_*` / `BACKEND` switches.

| Part | What you run | Marks |
|---|---|---|
| A | Five prompting strategies × 5 queries, 2-step chain, function-calling JSON | 5 |
| B | CoT, ToT, Step-Back, PAL, ReACT, Self-Reflection | 8 |
| C | ≥60 preference pairs, DPO on the Assignment 1 adapter, 5 held-out comparisons | 7 |
| **Total** | | **20** |

**Domain:** Medical & Clinical Literature — same corpus and QLoRA adapter as Assignment 1A.

**How generation is chosen.** If Assignment 1 wrote `outputs/qlora_adapters/adapter_B` and the machine has **CUDA or Apple MPS**, every cell uses that adapter. Otherwise the notebook uses a deterministic corpus-grounded **offline** backend so a CPU-only Run all still finishes. Offline text is a reproducible stand-in, not model output; every saved artefact records which backend produced it.

**Hardware.** One 1.5B model in memory at a time. SFT generations are unloaded before DPO loads. CUDA (T4) keeps the brief as written: 5-sample self-consistency, 5 APE variants, 3 DPO epochs. **Apple MPS** (this Mac's GPU) uses the same adapter path but fewer iterations so a local run stays near an hour: 3-sample self-consistency, 3 APE variants, ~20 DPO steps, 128-token generations. CPU DPO is 8 short LoRA steps if the adapter exists; Parts A/B stay offline on CPU.


In [ ]:
# Colab's "Open notebook from GitHub" copies only this .ipynb, not src/.
# Clone the repo into /content so later cells can import src/.
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/madhu-GG/bits_llm_assignment_1_sem3.git"
REPO_NAME = "bits_llm_assignment_1_sem3"

def _has_src(root: Path) -> bool:
    return (root / "src" / "runtime.py").exists() and (root / "src" / "prompt_strategies.py").exists()

if _has_src(Path.cwd()):
    print("Already in the assignment folder:", Path.cwd())
elif "google.colab" in sys.modules:
    dest = Path("/content") / REPO_NAME
    if not _has_src(dest):
        print("Cloning", REPO_URL, "into", dest)
        env = os.environ.copy()
        env["GIT_TERMINAL_PROMPT"] = "0"
        clone_url = REPO_URL
        token = None
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")  # optional Colab secret; never hardcode a token
        except Exception:
            token = None
        if token:
            clone_url = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
        subprocess.run(["git", "clone", "--depth", "1", clone_url, str(dest)], check=True, env=env)
        if token:
            subprocess.run(["git", "-C", str(dest), "remote", "set-url", "origin", REPO_URL], check=True)
    os.chdir(dest)
    print("Working directory:", Path.cwd())
else:
    raise FileNotFoundError(
        "src/ is missing. Open this notebook from the assignment folder, or clone the GitHub repo first."
    )

assert _has_src(Path.cwd()), f"src/ still missing under {Path.cwd()}"


In [ ]:
# Install once on Colab. Locally, use: pip install -r requirements.txt
import sys
if "google.colab" in sys.modules:
    %pip -q install pypdf pyarrow matplotlib transformers accelerate peft trl datasets bitsandbytes sentencepiece sacremoses psutil huggingface_hub pandas


In [ ]:
# Memory env vars must be set BEFORE torch is imported.
from __future__ import annotations

import json
import sys
import importlib
from pathlib import Path

from src.runtime import apply_memory_env, detect_runtime, memory_snapshot, project_root, seed_everything, training_budget

apply_memory_env()
seed_everything(7)

ROOT = project_root()
# Put src/ on sys.path so the Assignment 2 helpers import each other by module name.
for path in (str(ROOT.resolve()), str((ROOT / "src").resolve())):
    if path in sys.path:
        sys.path.remove(path)
    sys.path.insert(0, path)
sys.path_importer_cache.clear()
importlib.invalidate_caches()

from a2_paths import DEFAULT_SFT_ADAPTER, DPO_ADAPTER_DIR, PREFERENCE_DATASET, base_model_source, sft_adapter_path
from domain_lm import DomainLM, choose_generation_backend, wrap

RUNTIME = detect_runtime()
BUDGET = training_budget(RUNTIME)
SFT_ADAPTER = DEFAULT_SFT_ADAPTER
ADAPTER_DIR = sft_adapter_path(SFT_ADAPTER)
ARTIFACTS = ROOT / "outputs"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# CUDA or Apple MPS plus adapter_B → real generations.
BACKEND = choose_generation_backend(ADAPTER_DIR, RUNTIME)
SC_SAMPLES = int(BUDGET.get("a2_self_consistency_samples") or 5)
APE_VARIANTS = int(BUDGET.get("a2_ape_variants") or 5)
REACT_CYCLES = int(BUDGET.get("a2_react_cycles") or 5)
A2_NEW_TOKENS = int(BUDGET.get("a2_max_new_tokens") or 200)
lm = DomainLM(
    base_model_id=base_model_source(),
    adapter_path=ADAPTER_DIR if BACKEND == "adapter" else None,
    backend=BACKEND,
    label=f"sft_{SFT_ADAPTER}",
)
print(json.dumps({"runtime": RUNTIME, "budget_tier": BUDGET.get("gpu_tier"),
                  "adapter": str(ADAPTER_DIR), "adapter_exists": ADAPTER_DIR.exists(),
                  "sc_samples": SC_SAMPLES, "ape_variants": APE_VARIANTS,
                  "dpo_max_steps": BUDGET.get("dpo_max_steps"),
                  "lm": lm.describe()}, indent=2, default=str))
if BACKEND == "offline" and (RUNTIME.get("cuda") or RUNTIME.get("mps")) and not ADAPTER_DIR.exists():
    print("Accelerator is present but Assignment 1 adapter_B is missing. Parts A/B use the offline backend; run Assignment 1A first for the marked adapter run.")
memory_snapshot("a2_startup")


## Part A — Prompt paradigms & system design  [5 marks]

Load the Assignment 1 adapter (or the offline stand-in) and apply **five prompting strategies** to the same five domain queries. Decoding is greedy (`temperature=0`) except self-consistency at `0.7`. Token length and sample counts come from the runtime budget: CUDA keeps 200 tokens / 5 SC samples / 5 APE variants; MPS uses 128 / 3 / 3 so a local run stays near an hour.

| Strategy | Configuration | Expected effect |
|---|---|---|
| Zero-shot | No examples; task described in the instruction only | Tests raw adapter capability |
| Few-shot (3-shot) | 3 domain Q&A examples prepended | Improves format adherence |
| Instruction prompting | Explicit role + step-by-step instruction format | Improves structure |
| Self-consistency | Same prompt N× at `temperature=0.7`, majority vote (N=5 CUDA, N=3 MPS) | Reduces variance |
| APE | N LLM-generated prompt variants, best selected by output quality (N=5 CUDA, N=3 MPS) | Optimises instruction wording |

Answers are scored by one shared rubric (`src/answer_scoring.py`) out of 100: grounding 35, structure 20, completeness 20, specificity 15, non-repetition 10. The same scorer is reused for ToT / APE selection and for the chosen/rejected decision in Part C.


In [ ]:
import pandas as pd
import prompt_strategies as ps
from assignment2_queries import DOMAIN_QUERIES

# Decode length follows the runtime budget (200 tokens on CUDA, shorter on MPS/CPU).
ps.MAX_NEW_TOKENS = A2_NEW_TOKENS

def show(frame):
    try:
        display(frame)
    except Exception:
        print(frame.to_string(index=False))

for i, q in enumerate(DOMAIN_QUERIES, 1):
    print(f"Q{i}. {q}")


In [ ]:
# 5 queries × 5 strategies. CUDA: 5 SC samples / 5 APE variants. MPS: 3 / 3.
a1_rows = ps.run_all_strategies(lm, DOMAIN_QUERIES, sc_samples=SC_SAMPLES, ape_variants=APE_VARIANTS)
print(f"{len(a1_rows)} strategy runs = {len(DOMAIN_QUERIES)} queries × {len(ps.STRATEGIES)} strategies "
      f"(self-consistency n={SC_SAMPLES}, APE variants={APE_VARIANTS})")


In [ ]:
table = pd.DataFrame([{
    "query": f"Q{DOMAIN_QUERIES.index(r['query']) + 1}",
    "strategy": r["strategy"],
    "score": r["score"]["total"],
    "grounding": r["score"]["grounding"],
    "structure": r["score"]["structure"],
    "words": r["score"]["words"],
    "answer": " ".join(r["answer"].split())[:110] + "...",
} for r in a1_rows])
show(table.pivot(index="query", columns="strategy", values="score"))
show(pd.DataFrame(ps.strategy_summary(a1_rows)))


In [ ]:
# Full text for one query so the qualitative difference is visible, not just the score.
focus = DOMAIN_QUERIES[1]
for r in [r for r in a1_rows if r["query"] == focus]:
    print("=" * 100)
    print(r["strategy"], "| score", r["score"]["total"])
    print(wrap(r["answer"]))


In [ ]:
# Self-consistency detail: the samples, the vote, and the agreement rate.
sc = ps.run_self_consistency(lm, focus, samples=SC_SAMPLES)
print("agreement:", sc["agreement"], "| majority cluster size:", sc["majority_size"], f"/ {SC_SAMPLES}")
print("vote counts:", {str(k)[:40]: v for k, v in sc["vote_counts"].items()})
for i, sample in enumerate(sc["samples"], 1):
    print(f"\n--- sample {i} ---\n", wrap(sample[:400]))


In [ ]:
# APE detail: the generated prompt variants and the score of each resulting answer.
ape = ps.run_ape(lm, focus, num_variants=APE_VARIANTS)
print("variants tried:", ape["variants_tried"], "| scores:", ape["variant_scores"])
print("\nRaw LLM proposals:\n", wrap(ape["raw_proposals"]))
print("\nSelected prompt:\n", wrap(ape["selected_prompt"]))


### A1 write-up

Read the pivot table and the summary, then record:

1. **Zero-shot** is the floor: short, unstructured, no limiting condition, no caveat.
2. **Few-shot** mainly buys format adherence; grounding moves less than structure.
3. **Instruction prompting** is the cheapest structure win (one generation).
4. **Self-consistency** removes unlucky samples. Report the agreement rate — a high rate means the extra 5× cost is not justified.
5. **APE** wins when the hand-written instruction was not optimal; look at the spread of `variant_scores`.


## A2 — Prompt chaining and function calling  [2 marks]

**Chain:** Step 1 extracts facts from retrieved corpus context → Step 2 synthesises the answer *only* from those facts. The same complex query is also run as a single-step prompt so the gain is from decomposition, not from extra context.

**Function call:** the model must return JSON matching `{entity, value, unit, source_clause, confidence}`. Invalid JSON is fed back as a repair turn.


In [ ]:
from assignment2_queries import CHAIN_QUERY, CLINICAL_FACT_SCHEMA

chain = ps.run_prompt_chain(lm, CHAIN_QUERY)
print("COMPLEX QUERY:\n", wrap(CHAIN_QUERY))
print("\n--- STEP 1: extracted facts ---\n", wrap(chain["step1_facts"]))
print("\n--- STEP 2: synthesised answer ---\n", wrap(chain["chained_answer"]))
print("\n--- SINGLE-STEP BASELINE ---\n", wrap(chain["single_step_answer"]))


In [ ]:
comparison = pd.DataFrame([
    {"approach": "single-step prompt", **chain["single_step_score"]},
    {"approach": "2-step chain", **chain["chained_score"]},
])
show(comparison)
delta = chain["chained_score"]["total"] - chain["single_step_score"]["total"]
print(f"Chaining changes the rubric score by {delta:+.2f} points.")


In [ ]:
print(json.dumps(CLINICAL_FACT_SCHEMA, indent=2))
call = ps.run_function_call(lm, "Extract one clinical fact about diagnostic test performance.")
print("valid:", call["valid"], "| attempts:", len(call["attempts"]))
for attempt in call["attempts"]:
    print(f"\nattempt {attempt['attempt']} valid={attempt['valid']}"
          + (f" error={attempt.get('error')}" if not attempt["valid"] else ""))
    print(wrap(attempt["raw"][:400]))
print("\nParsed object:\n", json.dumps(call["parsed"], indent=2))


In [ ]:
# Negative control: the validator must reject malformed or off-schema output.
for bad in ['{"entity": "x"}', "not json at all",
            '{"entity":"x","value":"y","unit":"z","source_clause":"c","confidence":"high"}']:
    try:
        ps.parse_structured_output(bad)
        print("ACCEPTED (unexpected):", bad)
    except ps.SchemaValidationError as exc:
        print("rejected:", exc)


In [ ]:
artefact_a = {
    "backend": lm.describe(),
    "a1_rows": [{k: v for k, v in r.items() if k != "samples"} for r in a1_rows],
    "a1_summary": ps.strategy_summary(a1_rows),
    "a2_chain": chain,
    "a2_function_call": call,
}
path = ARTIFACTS / "part_a_results.json"
path.write_text(json.dumps(artefact_a, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("saved", path)
memory_snapshot("after_part_a")


## Part B — Structured & iterative reasoning  [8 marks]

Same five Part A queries plus two multi-step problems that need decomposition.

| Step | Deliverable | Marks |
|---|---|---|
| B1 | CoT, ToT, Step-Back × 2 domain problems + comparison table | 4 |
| B2 | PAL (executed Python) + ReACT (≥3 cycles) + Self-Reflection | 4 |


In [ ]:
import reasoning_strategies as rs
from assignment2_queries import PAL_PROBLEMS, REACT_QUESTION, REASONING_PROBLEMS

rs.MAX_NEW_TOKENS = A2_NEW_TOKENS

for p in REASONING_PROBLEMS:
    print(p["id"], "->", wrap(p["question"]), "\n")


## B1 — Chain-of-Thought

`Let's think step by step.` is appended; the model numbers each step and finishes with `Answer:`. The same query is also run with a plain prompt.

**When to use it:** any multi-step clinical query where the conclusion depends on chaining a definition to a limiting condition (dose selection, test interpretation, risk weighing).


In [ ]:
cot_rows = [rs.run_cot(lm, p["question"]) for p in REASONING_PROBLEMS]
cot_rows += [rs.run_cot(lm, q) for q in DOMAIN_QUERIES]
ids = [p["id"] for p in REASONING_PROBLEMS] + [f"Q{i}" for i in range(1, 6)]
show(pd.DataFrame([{
    "problem": ids[i],
    "standard": r["standard_score"]["total"],
    "cot": r["score"]["total"],
    "delta": round(r["score"]["total"] - r["standard_score"]["total"], 2),
    "steps": r["reasoning_steps"],
} for i, r in enumerate(cot_rows)]))
print("\nSTANDARD\n", wrap(cot_rows[0]["standard_answer"]))
print("\nCHAIN-OF-THOUGHT\n", wrap(cot_rows[0]["answer"]))


## B1 — Tree-of-Thought

Three candidate paths at `temperature=0.7`, each forced down a different lens (*mechanism*, *measurement*, *risk-and-monitoring*). The shared rubric picks the winner.

**When to use it:** a complex problem with genuine alternatives, e.g. deploying a test in two prevalence settings.


In [ ]:
tot_rows = [rs.run_tot(lm, p["question"]) for p in REASONING_PROBLEMS]
for problem, t in zip(REASONING_PROBLEMS, tot_rows):
    print("=" * 100)
    print(problem["id"], "| path scores:", t["path_scores"], "| selected:", t["selected_lens"])
    for p in t["paths"]:
        print(f"\n-- path {p['path']} ({p['lens']}) score={p['score']['total']}")
        print(wrap(p["text"][:600]))


## B1 — Step-Back prompting

The model first writes a general-principle question, answers that, then applies the principle to the specific case. Useful when a specific clinical question is really an instance of a measurement or evidence principle.


In [ ]:
sb_rows = [rs.run_stepback(lm, p["question"]) for p in REASONING_PROBLEMS]
for problem, s in zip(REASONING_PROBLEMS, sb_rows):
    print("=" * 100)
    print(problem["id"])
    print("step-back question:", wrap(s["stepback_question"]))
    print("principle:", wrap(s["principle"]))
    print("final:\n", wrap(s["answer"]))

show(pd.DataFrame([{
    "problem": problem["id"],
    "standard": cot_rows[i]["standard_score"]["total"],
    "cot": cot_rows[i]["score"]["total"],
    "tot_best": tot_rows[i]["score"]["total"],
    "step_back": sb_rows[i]["score"]["total"],
    "tot_paths": tot_rows[i]["path_scores"],
} for i, problem in enumerate(REASONING_PROBLEMS)]))


## B2 — PAL (Program-Aided Language model)

For calculation-type questions, prose arithmetic is unreliable. PAL makes the model emit Python; the **executed** value is the answer.

Model output is untrusted, so `safe_execute` AST-validates first: only `math` may be imported, and loops / dunders / `eval` / `open` are rejected.


In [ ]:
pal_rows = [rs.run_pal(lm, p) for p in PAL_PROBLEMS]
for problem, r in zip(PAL_PROBLEMS, pal_rows):
    print("=" * 100)
    print(problem["id"], "|", wrap(problem["question"]))
    print("\nGenerated program:\n" + str(r.get("code")))
    print(f"\nExecuted result: {r['result']} {r.get('unit', '')}"
          f" | expected {problem['expected']} | match={r['matches_expected']}")

show(pd.DataFrame([{
    "id": r["problem_id"], "result": r["result"], "unit": r.get("unit"),
    "expected": r["expected"], "matches": r["matches_expected"],
    "attempts": len(r["attempts"]),
} for r in pal_rows]))

print("\nSandbox rejections:")
for bad in ["import os\nresult = os.getcwd()",
            "result = __import__('os').listdir('.')",
            "result = open('secrets.txt').read()",
            "while True:\n    pass"]:
    try:
        rs.safe_execute(bad)
        print("ACCEPTED (unexpected!):", bad)
    except Exception as exc:
        print(f"rejected [{type(exc).__name__}]: {exc}")


## B2 — ReACT (Thought → Action → Observation)

Available actions: `search_corpus[...]`, `lookup_topic[...]`, `calculate[...]`, `finish[...]`. Each observation is real corpus text, not model memory. Assignment minimum is **3 cycles**.


In [ ]:
react = rs.run_react(lm, REACT_QUESTION, max_cycles=REACT_CYCLES)
print("QUESTION:\n", wrap(REACT_QUESTION))
print(f"\ncycles: {react['cycles']} (assignment minimum is 3; this run max={REACT_CYCLES})\n")
for step in react["trace"]:
    print("-" * 100)
    print(f"[cycle {step['cycle']}]")
    print("Thought:     ", wrap(step["thought"]))
    print("Action:      ", f"{step['action']}[{step.get('argument', '')}]")
    print("Observation: ", wrap(str(step["observation"])[:500]))
print("=" * 100)
print("FINAL ANSWER:\n", wrap(react["answer"]))
assert react["cycles"] >= 3, "ReACT produced fewer than 3 cycles"


## B2 — Self-Reflection

A draft is produced, critiqued for missing corpus facts / limiting conditions / monitoring, then revised. The revision is one of the `chosen` sources in Part C.


In [ ]:
reflect_rows = [rs.run_self_reflection(lm, p["question"]) for p in REASONING_PROBLEMS]
reflect_rows += [rs.run_self_reflection(lm, q) for q in DOMAIN_QUERIES]
show(pd.DataFrame([{
    "draft_score": r["draft_score"]["total"],
    "revised_score": r["score"]["total"],
    "delta": round(r["score"]["total"] - r["draft_score"]["total"], 2),
} for r in reflect_rows]))
print("\nDRAFT\n", wrap(reflect_rows[0]["draft"]))
print("\nREVISED\n", wrap(reflect_rows[0]["answer"]))


In [ ]:
artefact_b = {
    "backend": lm.describe(),
    "cot": cot_rows,
    "tot": tot_rows,
    "step_back": sb_rows,
    "pal": pal_rows,
    "react": react,
    "self_reflection": reflect_rows,
}
path = ARTIFACTS / "part_b_results.json"
path.write_text(json.dumps(artefact_b, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("saved", path)
memory_snapshot("after_part_b")


## Part C — Preference alignment via DPO  [7 marks]

| Step | Deliverable | Marks |
|---|---|---|
| C1 | ≥60 pairs (`prompt`, `chosen`, `rejected`) + counts and length delta | 2 |
| C2 | DPO, `beta=0.1`, 3 epochs, loss + reward margin, `outputs/dpo_adapter/` | 3 |
| C3 | 5 unseen queries, SFT vs DPO table, 100–150 word analysis | 2 |

- `chosen` — CoT / instruction / self-consistency / self-reflection / APE
- `rejected` — zero-shot single-pass for the **same** prompt
- A pair is kept only when the shared rubric prefers `chosen` by ≥3 points
- The five C3 queries are excluded from the training pool


In [ ]:
import preference_data as pdata

# Build preference pairs from the Part A/B answers already in memory.
pairs, skipped = pdata.harvest_pairs(a1_rows, extra_rows=cot_rows + reflect_rows)
print(f"harvested {len(pairs)} pairs from Parts A/B (skipped {skipped} low-margin)")

if len(pairs) < 60:
    used = {p["prompt"] for p in pairs}
    remaining = [q for q in pdata.prompt_pool() if q not in used]
    # CoT, instruction, few-shot, and self-reflection answers fill the ≥60-pair requirement.
    extra, extra_stats = pdata.build_preference_pairs(lm, queries=remaining, min_pairs=0, cheap=True)
    pairs.extend(extra)
    print("filled with", extra_stats)

if len(pairs) < 60:
    used = {p["prompt"] for p in pairs}
    remaining = [q for q in pdata.prompt_pool() if q not in used]
    extra, extra_stats = pdata.build_preference_pairs(lm, queries=remaining or pdata.prompt_pool(), min_pairs=0, cheap=False)
    pairs.extend(extra)
    print("full-generator fill", extra_stats)

stats = pdata.summarise(pairs)
stats["skipped_low_margin"] = skipped
stats["meets_minimum"] = len(pairs) >= 60
pdata.save_jsonl(pairs, PREFERENCE_DATASET)
problems = pdata.validate(pairs, min_pairs=60)
print("validation problems:", problems or "none")
assert not problems, problems
print(json.dumps(stats, indent=2))
print(f"\n{stats['pairs']} pairs over {stats['unique_prompts']} prompts")
print(f"avg chosen {stats['avg_chosen_words']} words, rejected {stats['avg_rejected_words']} words, "
      f"delta {stats['avg_length_delta_words']:+} words")
show(pd.DataFrame([{"chosen_source": k, "pairs": v} for k, v in stats["pairs_by_chosen_source"].items()]))
sample = pairs[0]
print("\nPROMPT:\n", wrap(sample["prompt"]))
print("\nREJECTED (zero-shot):\n", wrap(sample["rejected"]))
print("\nCHOSEN (" + sample["chosen_source"] + "):\n", wrap(sample["chosen"]))


**Length-delta caveat.** Chosen answers are systematically longer. DPO can learn "longer is better". Two guards: the rubric caps completeness at 220 words, and C3 flags a row as *not improved* when the DPO answer is much longer without a gain in grounding or structure.


## C2 — DPO fine-tuning

The Assignment 1 adapter is the SFT policy. TRL uses the frozen base+adapter as the implicit reference (`ref_model=None`). `beta=0.1`. **CUDA:** 3 epochs. **Apple MPS:** about 20 LoRA steps at a short sequence so a local hour is realistic. **CPU:** 8 LoRA steps if `adapter_B` exists. There is no `RUN_DPO` flag.


In [ ]:
import dpo_training as dpo

print(json.dumps(dpo.DPO_CONFIG, indent=2))
print("SFT adapter:", ADAPTER_DIR, "| exists:", ADAPTER_DIR.exists())
print("DPO output :", DPO_ADAPTER_DIR)
print("DPO budget:", {k: BUDGET.get(k) for k in ("gpu_tier", "dpo_epochs", "dpo_max_steps", "dpo_max_length")})

run_dpo = bool(ADAPTER_DIR.exists() and BUDGET.get("can_load_model", True))
log_history = []
if run_dpo:
    # Free the generation model before DPO loads.
    lm.close()
    result = dpo.train_dpo(PREFERENCE_DATASET, sft_adapter=SFT_ADAPTER, output_dir=DPO_ADAPTER_DIR)
    log_path = DPO_ADAPTER_DIR / "dpo_log_history.json"
    log_history = json.loads(log_path.read_text(encoding="utf-8")) if log_path.exists() else result.get("history", [])
    print(json.dumps(result["config"], indent=2))
else:
    log_path = DPO_ADAPTER_DIR / "dpo_log_history.json"
    log_history = json.loads(log_path.read_text(encoding="utf-8")) if log_path.exists() else []
    why = "adapter_B not found. Run Assignment 1A first." if not ADAPTER_DIR.exists() else BUDGET.get("reason")
    print("DPO training skipped:", why)


In [ ]:
import matplotlib.pyplot as plt

if log_history:
    epochs = dpo.epoch_metrics(log_history)
    show(pd.DataFrame(epochs))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot([e["epoch"] for e in epochs], [e["dpo_loss"] for e in epochs], marker="o")
    axes[0].set_title("DPO loss per epoch")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("loss")
    axes[1].plot([e["epoch"] for e in epochs], [e["reward_chosen"] for e in epochs], marker="o", label="chosen")
    axes[1].plot([e["epoch"] for e in epochs], [e["reward_rejected"] for e in epochs], marker="o", label="rejected")
    axes[1].plot([e["epoch"] for e in epochs], [e["reward_margin"] for e in epochs], marker="s", linestyle="--", label="margin")
    axes[1].set_title("Implicit rewards per epoch")
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    plt.tight_layout()
    fig.savefig(ARTIFACTS / "dpo_curves.png", dpi=140)
    plt.show()
    plt.close(fig)
    print("DPO loss should decrease across epochs; the reward margin should increase.")
else:
    show(pd.DataFrame([
        {"epoch": 1, "dpo_loss": None, "reward_chosen": None, "reward_rejected": None,
         "reward_margin": None, "reward_accuracy": None},
        {"epoch": 2, "dpo_loss": None, "reward_chosen": None, "reward_rejected": None,
         "reward_margin": None, "reward_accuracy": None},
        {"epoch": 3, "dpo_loss": None, "reward_chosen": None, "reward_rejected": None,
         "reward_margin": None, "reward_accuracy": None},
    ]))
    print("No DPO log yet. This table fills after C2 trains.")


## C3 — Alignment evaluation on five unseen queries

Both adapters answer the same **bare zero-shot** prompt. Alignment should produce the preferred shape *without* scaffolding. On T4 and Apple MPS the SFT generations are done first, the model is freed, then the DPO adapter is loaded — never two 1.5B copies.


In [ ]:
from assignment2_queries import HELD_OUT_QUERIES

for i, q in enumerate(HELD_OUT_QUERIES, 1):
    print(f"{i}. {q}")
assert not set(q.lower() for q in HELD_OUT_QUERIES) & set(p["prompt"].lower() for p in pairs)
print("\nconfirmed: none of these queries appears in the preference dataset")


In [ ]:
# Generate with the SFT adapter, free it, then generate with the DPO adapter.
dpo_dir = DPO_ADAPTER_DIR
use_real_sft = bool(
    (RUNTIME.get("cuda") or RUNTIME.get("mps"))
    and ADAPTER_DIR.exists()
    and BUDGET.get("can_load_model", True)
)
use_real_dpo = bool(
    (RUNTIME.get("cuda") or RUNTIME.get("mps"))
    and dpo_dir.exists()
    and ((dpo_dir / "adapter_config.json").exists() or (dpo_dir / "config.json").exists())
)

if use_real_sft and BACKEND == "adapter" and lm._model is not None:
    sft_lm = lm
elif use_real_sft:
    sft_lm = DomainLM(
        base_model_id=base_model_source(),
        adapter_path=ADAPTER_DIR,
        backend="adapter",
        label=f"sft_{SFT_ADAPTER}",
    )
else:
    sft_lm = DomainLM(backend="offline", label=f"sft_{SFT_ADAPTER}")
sft_outputs = dpo.generate_zero_shot(sft_lm, HELD_OUT_QUERIES, max_new_tokens=A2_NEW_TOKENS)
sft_desc = sft_lm.describe()
sft_lm.close()

if use_real_dpo:
    dpo_lm = DomainLM(base_model_id=base_model_source(), adapter_path=dpo_dir,
                      backend="adapter", label="dpo_adapter")
else:
    # Offline stand-in: a bare prompt is answered in the preferred (aligned) shape.
    dpo_lm = DomainLM(backend="offline", label="dpo_adapter", aligned=True)
dpo_outputs = dpo.generate_zero_shot(dpo_lm, HELD_OUT_QUERIES, max_new_tokens=A2_NEW_TOKENS)
dpo_desc = dpo_lm.describe()
dpo_lm.close()

rows_c3 = dpo.compare_outputs(sft_outputs, dpo_outputs, HELD_OUT_QUERIES)
summary = dpo.comparison_summary(rows_c3)
print(json.dumps({"sft": sft_desc, "dpo": dpo_desc, "summary": summary}, indent=2))


In [ ]:
for r in rows_c3:
    print("=" * 100)
    print(f"Query {r['n']}: {r['query']}")
    print(f"\n[SFT {r['sft_score']['total']}]\n", wrap(r["sft_output"]))
    print(f"\n[DPO {r['dpo_score']['total']}]\n", wrap(r["dpo_output"]))
    print(f"\nImprovement: {r['improved']} — {r['reason']}")

show(pd.DataFrame([{
    "Query": f"Query {r['n']}",
    "SFT score": r["sft_score"]["total"],
    "DPO score": r["dpo_score"]["total"],
    "Delta": round(r["dpo_score"]["total"] - r["sft_score"]["total"], 2),
    "Improvement?": r["improved"],
    "Reason": r["reason"],
} for r in rows_c3]))

table_md = dpo.markdown_table(rows_c3)
(ARTIFACTS / "part_c_comparison.md").write_text(table_md, encoding="utf-8")
print(table_md)


In [ ]:
artefact_c = {
    "backend_sft": sft_desc,
    "backend_dpo": dpo_desc,
    "c1_stats": stats,
    "c2_config": dpo.DPO_CONFIG,
    "c2_epoch_metrics": dpo.epoch_metrics(log_history) if log_history else [],
    "c3_rows": rows_c3,
    "c3_summary": summary,
}
path = ARTIFACTS / "part_c_results.json"
path.write_text(json.dumps(artefact_c, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("saved", path)
memory_snapshot("done")


## C3 — Alignment analysis (100–150 words)

> **Rewrite this paragraph from your own GPU run before submitting.** Cite the row numbers and scores printed above.

DPO alignment improved domain response quality on the held-out set: mean rubric score rose from the SFT adapter to the DPO adapter, and the gain came almost entirely from *shape* rather than from new knowledge. The largest improvements were on Query 1 and Query 3, where the SFT adapter returned a single definitional sentence and the DPO adapter returned the definition plus the governing considerations, the limiting condition, and the educational caveat — exactly the pattern carried by the `chosen` responses. The weakest case was the query whose answer the SFT adapter already covered well; there the DPO output was longer without adding corpus content, which is the expected length-bias failure mode of preference training. DPO therefore transferred the *format and caveat discipline* of CoT and self-reflection into the weights, but it did not add factual coverage that was absent from the Assignment 1 corpus.

### Caveats

1. Preference pairs were generated and labelled by the same rubric that scores C3, so the evaluation is favourable by construction. An independent human rating of a 20-row sample is the honest check.
2. `chosen` is systematically longer than `rejected`, so some of the gain is length bias.
3. ~60–95 pairs over 19 prompts is a small, narrow dataset; do not extrapolate beyond this corpus.
4. All medical text is educational. It is not patient-specific clinical advice.

## Deliverables under `outputs/` and the repo root

| File | Step |
|---|---|
| `outputs/part_a_results.json` | A |
| `outputs/part_b_results.json` | B |
| `preference_dataset.jsonl` | C1 (≥60 pairs) |
| `outputs/dpo_adapter/` | C2 (GPU run) |
| `outputs/part_c_comparison.md`, `outputs/part_c_results.json` | C3 |
